# Управляемость

In [28]:
import numpy as np
import matplotlib.pyplot as plt
import control as ct
import scipy
from IPython.display import display
from sympy import Matrix

## Пример 1. Полностью управляемая система

In [29]:
A = np.array([[0, 1],
              [-2, -3]])
B = np.array([[0],
              [1]])
C = np.array([[1, 0]])
D = np.array([[0]])

G = ct.ss(A, B, C, D)

# проверка управляемости
U = ct.ctrb(A, B)
rU = np.linalg.matrix_rank(U)

print("Матрица управляемости U =")
display(Matrix(U))
print("rank(U) =", rU)

if rU == A.shape[0]:
    Gc, T = ct.reachable_form(G)
    Ac, Bc, Cc, Dc = Gc.A, Gc.B, Gc.C, Gc.D

    print("T =")
    display(T)

    print("Ac =")
    display(Matrix(Ac))

    print("Bc =")
    display(Matrix(Bc))

    print("Cc =")
    display(Matrix(Cc))

    print("Dc =")
    display(Matrix(Dc))
else:
    print("Система не является полностью управляемой.")

Матрица управляемости U =


Matrix([
[0.0,  1.0],
[1.0, -3.0]])

rank(U) = 2
T =


array([[0., 1.],
       [1., 0.]])

Ac =


...

Matrix([
[1.0],
[0.0]])

Cc =


Matrix([[0.0, 1.0]])

Dc =


Matrix([[0.0]])

## Пример 2. Неуправляемая система

In [30]:
#формирование системы
A2 = np.array([[0, 1, 0, 0],
               [-2, -3, 0, 0],
               [0, 0, 1, 0],
               [0, 0, 0, 2]])

B2 = np.array([[0],
               [1],
               [0],
               [0]])

C2 = np.array([[1, 0, 1, 0]])
D2 = np.array([[0]])

G2 = ct.ss(A2, B2, C2, D2)

In [31]:
# матрица управляемости
U2 = ct.ctrb(G2.A, G2.B)
rU2 = np.linalg.matrix_rank(U2)

print("Матрица управляемости U2 =")
display(Matrix(U2))

print("rank(U2) =", rU2)
print("Размерность системы =", A2.shape[0])

if rU2 == G2.A.shape[0]:
    print("Система полностью управляемая.")
else:
    print("Система неуправляемая.")

Матрица управляемости U2 =


Matrix([
[0.0,  1.0, -3.0,   7.0],
[1.0, -3.0,  7.0, -15.0],
[0.0,  0.0,  0.0,   0.0],
[0.0,  0.0,  0.0,   0.0]])

rank(U2) = 2
Размерность системы = 4
Система неуправляемая.


### Декомпозиция по управлению

In [32]:
def ctrbf_like(G, tol=None):
    """
    возвращает:
        Ab, Bb, Cb, T, k
    где
        Ab, Bb, Cb -- матрицы системы в новых координатах,
        T          -- матрица перехода к новым координтам y = T x,
        k          -- размерность управляемого подпространства.
    """
    A = np.asarray(G.A, dtype=float)
    B = np.asarray(G.B, dtype=float)
    C = np.asarray(G.C, dtype=float)

    n = A.shape[0]
    if G.D is None:
        D = np.zeros((C.shape[0], B.shape[1]))
    else:
        D = np.asarray(G.D, dtype=float)

    # 1) матрица управляемости
    U = ct.ctrb(A, B)

    # 2) ранг
    if tol is None:
        tol = max(U.shape) * np.linalg.norm(U, 2) * np.finfo(float).eps

    # разложение QR по U: первые k столбцов Q дают базис Im(U)
    Qc, Rc, piv = scipy.linalg.qr(U, mode="economic", pivoting=True)
    diagR = np.abs(np.diag(Rc))
    k = int(np.sum(diagR > tol))

    # базис управляемого подпространства и его ортогонального дополнения
    Vc = Qc[:, :k]          # controllable basis
    Vuc = Qc[:, k:]         # uncontrollable complement

    # ортогональная матрица P = [Vuc Vc], x = P [x_uc; x_c]
    P = np.hstack([Vuc, Vc])
    
    # x = P y, откуда y = P^T x
    T = P.T

    # 3) преобразование матриц системы
    Ab = T @ A @ np.linalg.inv(T)
    Bb = T @ B
    Cb = C @ np.linalg.inv(T)

    return Ab, Bb, Cb, T, k

In [34]:
Ab, Bb, Cb, T, k = ctrbf_like(G2)
print("T =")
display(T)
print("Ac =")
display(Matrix(Ab))
print("Bc =")
display(Matrix(Bb))
print("Cc =")
display(Matrix(Cb))
print("Dc =")
display(Matrix(G2.D))

T =


array([[ 0.        ,  0.        ,  1.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  1.        ],
       [-0.42288547,  0.90618314, -0.        , -0.        ],
       [ 0.90618314,  0.42288547,  0.        ,  0.        ]])

Ac =


Matrix([
[1.0, 0.0,                0.0,                0.0],
[0.0, 2.0,                0.0,                0.0],
[0.0, 0.0,  -2.08029197080292,  -2.97080291970803],
[0.0, 0.0, 0.0291970802919707, -0.919708029197081]])

Bc =


Matrix([
[              0.0],
[              0.0],
[0.906183139995265],
[0.422885465331124]])

Cc =


Matrix([[1.0, 0.0, -0.422885465331124, 0.906183139995265]])

Dc =


Matrix([[0.0]])

### Пример из текста методички

Управляемость при $\delta = 10\varepsilon$: аналитически $\det U = -\delta^3 \neq 0$ (система вполне управляема), а численный ранг матрицы управляемости равен 1.

In [ ]:
eps = np.finfo(float).eps
delta = 10 * eps

A = np.array([[1, delta],
              [0, 1]])
B = np.array([[1],
              [delta]])
G = ct.ss(A, B, [[1, 0]], [[0]])

U = ct.ctrb(A, B)
print("U =\n", U)
print("сингулярные числа U:", np.linalg.svd(U, compute_uv=False))
print("численный ранг:", np.linalg.matrix_rank(U))

# каноническая форма существует только для управляемой системы
try:
    Gc, T = ct.reachable_form(G)
except ValueError as err:
    print("Каноническая форма не построена:", err)

### Пример из текста методички

Функция `ctrbf_like` в редакции пособия: матрица $T$ ортогональна, поэтому $T^{-1} = T^T$; порядок блоков [неуправляемая часть; управляемая часть] совпадает с `ctrbf` в MATLAB.

In [ ]:
def ctrbf_like_book(G, tol=None):
    """
    Декомпозиция по управлению (аналог ctrbf в MATLAB).
    Возвращает Ab, Bb, Cb, T, k, где
      Ab, Bb, Cb -- матрицы системы в координатах xb = T x,
      T          -- ортогональная матрица перехода,
      k          -- размерность управляемого подпространства.
    Первые n-k координат xb неуправляемы, последние k -- управляемы.
    """
    A = np.asarray(G.A, dtype=float)
    B = np.asarray(G.B, dtype=float)
    C = np.asarray(G.C, dtype=float)

    # 1) матрица управляемости
    U = ct.ctrb(A, B)

    # 2) численный ранг: порог, как в np.linalg.matrix_rank
    if tol is None:
        tol = max(U.shape) * np.linalg.norm(U, 2) * np.finfo(float).eps

    # QR с выбором ведущего столбца: первые k столбцов Q -- базис Im(U)
    Q, R, piv = scipy.linalg.qr(U, pivoting=True)
    k = int(np.sum(np.abs(np.diag(R)) > tol))

    # 3) базис: [неуправляемая часть, управляемая часть]
    P = np.hstack([Q[:, k:], Q[:, :k]])   # x = P xb
    T = P.T                               # P ортогональна: P^{-1} = P^T

    # 4) преобразование матриц системы
    Ab = T @ A @ T.T
    Bb = T @ B
    Cb = C @ T.T
    return Ab, Bb, Cb, T, k


Ab, Bb, Cb, T, k = ctrbf_like_book(G2)
n = G2.A.shape[0]
print("k =", k)
display(Matrix(np.round(Ab, 4)))
# проверка блочной структуры: верхний правый блок Ab и первые n-k строк Bb нулевые
print("max|Ab[:n-k, n-k:]| =", np.abs(Ab[:n - k, n - k:]).max())
print("max|Bb[:n-k]|      =", np.abs(Bb[:n - k]).max())

### Пример из текста методички

Наблюдаемость: матрица наблюдаемости и декомпозиция двойственной системы $(A^T, C^T)$.

In [ ]:
N = ct.obsv(G2.A, G2.C)
print("rank N =", np.linalg.matrix_rank(N), "из", G2.A.shape[0])

# наблюдаемость (A, C) <=> управляемость (A^T, C^T)
G2_dual = ct.ss(G2.A.T, G2.C.T, G2.B.T, G2.D.T)
_, _, _, _, k_obs = ctrbf_like_book(G2_dual)
print("размерность наблюдаемого подпространства:", k_obs)